# Baseline: RM encoder + Dumer decoder (plain Dumer, no neural network)

This is the reference decoder used as **"Dumer"** in the paper and in the `HiKO_PermSelector-*` notebooks.

* **Encoder:** Reed-Muller code RM(m, r) by the recursive Plotkin construction (u, u⊕v).
* **Decoder:** Dumer's recursive decoder: exact LSE (box-plus) combination for the v-branch, hard re-encoded feedback for the u-branch,
  MAP decoding at the leaves (repetition code, and first-order RM codes through the fast Hadamard transform).
* **Channel:** BPSK over AWGN, SNR defined as $E_s/N_0 = 1/(2\sigma^2)$ (the convention of the KO code base).

`NeuralRM(r, m)` with its default arguments is exactly this classical decoder (its neural parts are switched off).
Cell 2 holds all parameters. In the paper's notation the code is RM(m, r), so `M = 8, R = 3` is RM(8,3).

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Select GPU index (project rule: GPU 1 only)

import sys, math, time, json, random
from math import comb
from pathlib import Path
from datetime import datetime
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

In [ ]:
# ========================================
# ALL PARAMETERS & CONFIGURATION
# ========================================
MODE = "compute_and_save"        # "compute_and_save" | "analyze_only"
ANALYZE_RUN_DIR = ""             # used only when MODE == "analyze_only" (path to a run directory)

PROJECT_NAME = "RM_Dumer_baseline"
DATASET_NAME = "awgn_bpsk"
MODEL_NAME = "plain_dumer"

# --- Code: RM(M, R) in the paper's notation, n = 2^M, order R ---
R = 3
M = 8

# --- Evaluation ---
EVAL_SNRS = [-2.0, -1.5, -1.0, -0.5, 0.0, 0.5]     # Es/N0 in dB
MIN_BLOCK_ERRORS = 400           # stop an SNR point after this many block errors ...
MAX_WORDS = 10_000_000           # ... or this many codewords
BATCH = 2000                     # frames per GPU batch (use 1000 for n = 512)

# --- Sanity check against the RM baseline numbers printed in the paper (BER); set to {} to skip ---
PAPER_REF_BER = {(8, 3, 0.0): 8.37e-4, (8, 3, -5.0): 4.60e-1}     # (m, r, SNR dB): BER
CHECK_WORDS = 400_000

SEED = 42
NUM_GPUS = torch.cuda.device_count()
PLOT_DPI = 300

print("=" * 70); print("CONFIGURATION"); print("=" * 70)
for k in ["MODE", "R", "M", "EVAL_SNRS", "MIN_BLOCK_ERRORS", "MAX_WORDS", "BATCH", "SEED"]:
    print(f"{k}: {globals()[k]}")
print(f"Available GPUs (visible): {NUM_GPUS}"); print("=" * 70)

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    print(f"✓ Seeds set to {seed}")

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ========================================
# AUTO DIRECTORY SETUP (name built from the Cell-2 parameters)
# ========================================
def generate_run_name_from_config():
    p = {'r': R, 'm': M, 'minerr': MIN_BLOCK_ERRORS, 'maxw': MAX_WORDS, 'seed': SEED}
    return PROJECT_NAME + "_" + "_".join(f"{k}{v}" for k, v in p.items()) + "_" + datetime.now().strftime("%Y%m%d_%H%M%S")

if MODE == "analyze_only":
    RESULTS_BASE_DIR = Path(ANALYZE_RUN_DIR); RUN_NAME = RESULTS_BASE_DIR.name
else:
    RUN_NAME = generate_run_name_from_config(); RESULTS_BASE_DIR = Path("ResultsAndWeights") / RUN_NAME
RESULTS_DIR = RESULTS_BASE_DIR / "results_data"
if MODE == "compute_and_save":
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("=" * 70); print("DIRECTORY SETUP"); print("=" * 70)
print(f"Run Name: {RUN_NAME}"); print(f"Results Base: {RESULTS_BASE_DIR.absolute()}"); print("=" * 70)

In [ ]:
# ========================================
# UTILITY FUNCTIONS
# ========================================
CONFIG_VARS = ['MODE', 'PROJECT_NAME', 'R', 'M', 'EVAL_SNRS', 'MIN_BLOCK_ERRORS', 'MAX_WORDS', 'BATCH', 'SEED']

def save_results(metrics_dict, results_text, results_dir):
    sio.savemat(results_dir / "results_metrics.mat", metrics_dict)
    print(f"✓ Metrics saved (MATLAB .mat): {results_dir / 'results_metrics.mat'}")
    with open(results_dir / "results_summary.txt", 'w') as f:
        f.write(results_text)
    print(f"✓ Results saved (text format): {results_dir / 'results_summary.txt'}")

def log_all_config(base_dir):
    cfg = {k: globals()[k] for k in CONFIG_VARS if k in globals()}
    cfg.update(timestamp=datetime.now().isoformat(), cuda_available=torch.cuda.is_available())
    with open(base_dir / "config.json", 'w') as f:
        json.dump(cfg, f, indent=2)
    print(f"✓ Configuration saved: {base_dir / 'config.json'}")

## The baseline code: RM encoder + Dumer decoder
The cell below is the classical backbone. Read `NeuralRM._enc` (Plotkin encoder), `NeuralRM._dec` (Dumer recursion), `boxplus` (exact LSE), and `first_order_decode` (MAP decoding of first-order leaves with the fast Hadamard transform `fht`). The `ResMLP` class and the `enc_mode`/`dec_mode` options belong to the neural variants and are **not used** here.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from math import comb


def rm_dim(r, m):
    r = min(r, m)
    return sum(comb(m, i) for i in range(r + 1)) if r >= 0 else 0


def snr_db2sigma(snr_db):
    # Es/N0 convention, same as the existing KO / HiKO code
    return 10 ** (-snr_db / 20) / math.sqrt(2)


# ------------------------------------------------------------------ FHT
def fht(x):
    """Sylvester Hadamard transform along last dim (natural order)."""
    n = x.shape[-1]
    shp = x.shape[:-1]
    h = 1
    while h < n:
        x = x.reshape(*shp, n // (2 * h), 2, h)
        a, b = x[..., 0, :], x[..., 1, :]
        x = torch.stack((a + b, a - b), dim=-2).reshape(*shp, n)
        h *= 2
    return x


def first_order_encode(msg, m):
    """msg (B, m+1) +/-1 -> codeword (B, 2^m): x_j = s0 * prod_i s_i^{bit_i(j)}"""
    n = 2 ** m
    j = torch.arange(n, device=msg.device)
    x = msg[:, :1].expand(-1, n).clone()
    for i in range(m):
        bit = ((j >> i) & 1).bool()
        x = torch.where(bit.unsqueeze(0), x * msg[:, 1 + i:2 + i], x)
    return x


def first_order_decode(L, m, exact=True):
    """Bitwise MAP (exact or max-log) for RM(1,m). Returns msg LLRs (B, m+1)."""
    B = L.shape[0]
    T = fht(L) * 0.5                      # metric of +H_a codeword is T[a]
    out = []
    if exact:
        S = torch.logaddexp(T, -T)        # log(e^T + e^-T)
        out.append(torch.logsumexp(T, 1) - torch.logsumexp(-T, 1))
        for i in range(m):
            Sv = S.reshape(B, 2 ** (m - 1 - i), 2, 2 ** i)
            out.append(torch.logsumexp(Sv[:, :, 0, :].reshape(B, -1), 1)
                       - torch.logsumexp(Sv[:, :, 1, :].reshape(B, -1), 1))
    else:
        S = T.abs()
        out.append(T.max(1)[0] + T.min(1)[0])
        for i in range(m):
            Sv = S.reshape(B, 2 ** (m - 1 - i), 2, 2 ** i)
            out.append(Sv[:, :, 0, :].reshape(B, -1).max(1)[0]
                       - Sv[:, :, 1, :].reshape(B, -1).max(1)[0])
    return torch.stack(out, 1)


def boxplus(a, b, exact=True):
    if exact:
        # 2 atanh(tanh(a/2) tanh(b/2)), numerically stable form
        return (torch.sign(a) * torch.sign(b) * torch.minimum(a.abs(), b.abs())
                + torch.log1p(torch.exp(-(a + b).abs())) - torch.log1p(torch.exp(-(a - b).abs())))
    return torch.sign(a) * torch.sign(b) * torch.minimum(a.abs(), b.abs())


# ------------------------------------------------------------------ NN blocks
class ResMLP(nn.Module):
    """Coordinate-wise MLP whose output is ADDED to a classical function.
    Last layer zero-initialised -> module starts exactly at classical."""

    def __init__(self, d_in, hidden=32, layers=3, d_out=1, act='selu'):
        super().__init__()
        dims = [d_in] + [hidden] * (layers - 1)
        self.hid = nn.ModuleList([nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:])])
        self.out = nn.Linear(dims[-1], d_out)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)
        self.act = {'selu': F.selu, 'elu': F.elu, 'gelu': F.gelu, 'tanh': torch.tanh}[act]

    def forward(self, x):
        for l in self.hid:
            x = self.act(l(x))
        return self.out(x)


# ------------------------------------------------------------------ code tree
class Node:
    def __init__(self, r, m, path):
        self.r, self.m, self.path = r, m, path
        self.n, self.k = 2 ** m, rm_dim(r, m)
        if r == 0 or r == 1:
            self.leaf = True
        else:
            self.leaf = False
            self.u = Node(min(r, m - 1), m - 1, path + 'u')
            self.v = Node(r - 1, m - 1, path + 'v')

    def internal(self):
        if self.leaf:
            return []
        return [self] + self.u.internal() + self.v.internal()


class NeuralRM(nn.Module):
    """
    enc_mode : 'rm' (fixed RM code) | 'neural' (KO-style learned g)
    dec_mode : 'dumer' (classical) | 'neural' (learned fv, fu)
    """

    def __init__(self, r, m, enc_mode='rm', dec_mode='dumer', hidden=32, layers=3,
                 exact=True, feedback='hard', g_min_m=1, f_min_m=1, act='selu',
                 fu_full=True):
        super().__init__()
        self.r, self.m = r, m
        self.root = Node(r, m, 'R')
        self.n, self.k = self.root.n, self.root.k
        self.enc_mode, self.dec_mode = enc_mode, dec_mode
        self.exact, self.feedback = exact, feedback
        self.fu_full = fu_full
        self.g = nn.ModuleDict()
        self.fv = nn.ModuleDict()
        self.fu = nn.ModuleDict()
        for nd in self.root.internal():
            if enc_mode == 'neural' and nd.m >= g_min_m:
                self.g[nd.path] = ResMLP(2, hidden, layers, act=act)
            if dec_mode == 'neural' and nd.m >= f_min_m:
                self.fv[nd.path] = ResMLP(4, hidden, layers, act=act)
                self.fu[nd.path] = ResMLP(5 if fu_full else 3, hidden, layers, act=act)

    # ---------------- encoder
    def _enc(self, msg, nd):
        if nd.leaf:
            if nd.r == 0:
                return msg.expand(-1, nd.n)
            return first_order_encode(msg, nd.m)
        ku = nd.u.k
        u = self._enc(msg[:, :ku], nd.u)
        v = self._enc(msg[:, ku:], nd.v)
        uv = u * v
        if nd.path in self.g:
            uv = uv + self.g[nd.path](torch.stack((u, v), -1)).squeeze(-1)
        return torch.cat((u, uv), 1)

    def encode(self, msg):
        x = self._enc(msg, self.root)
        if self.enc_mode == 'neural':
            x = F.normalize(x, p=2, dim=1) * math.sqrt(self.n)
        return x

    # ---------------- decoder
    def _fb(self, llr):
        """turn message LLRs into the feedback symbols used for re-encoding"""
        if self.feedback == 'hard':
            return torch.sign(llr)
        if self.feedback == 'ste':        # hard forward, soft gradient
            s = torch.tanh(llr / 2)
            return s + (torch.sign(llr) - s).detach()
        return torch.tanh(llr / 2)        # 'soft'

    def _dec(self, L, nd, genie_msg=None):
        """returns (msg_llr (B,k), x_hat (B,n) re-encoded estimate)"""
        if nd.leaf:
            if getattr(self, 'collect', None) is not None:
                self.collect.append((nd, L))
            if nd.r == 0:
                llr = L.sum(1, keepdim=True)
            else:
                llr = first_order_decode(L, nd.m, self.exact)
            fb = genie_msg if genie_msg is not None else self._fb(llr)
            return llr, self._enc(fb, nd)
        h = nd.n // 2
        L1, L2 = L[:, :h], L[:, h:]
        Lv = boxplus(L1, L2, self.exact)
        if nd.path in self.fv:
            feat = torch.stack((torch.asinh(L1) / 3, torch.asinh(L2) / 3, torch.tanh(L1 / 2), torch.tanh(L2 / 2)), -1)
            Lv = Lv + self.fv[nd.path](feat).squeeze(-1)
        ku = nd.u.k
        gv = genie_msg[:, ku:] if genie_msg is not None else None
        gu = genie_msg[:, :ku] if genie_msg is not None else None
        llr_v, v_hat = self._dec(Lv, nd.v, gv)
        Lu = L1 + v_hat * L2
        if nd.path in self.fu:
            if self.fu_full:
                feat = torch.stack((torch.asinh(L1) / 3, torch.asinh(L2) / 3, v_hat, torch.tanh(L1 / 2), torch.tanh(L2 / 2)), -1)
            else:
                feat = torch.stack((torch.asinh(L1) / 3, torch.asinh(L2) / 3, v_hat), -1)
            Lu = Lu + self.fu[nd.path](feat).squeeze(-1)
        llr_u, u_hat = self._dec(Lu, nd.u, gu)
        uv = u_hat * v_hat
        if nd.path in self.g:
            uv = uv + self.g[nd.path](torch.stack((u_hat, v_hat), -1)).squeeze(-1)
        return torch.cat((llr_u, llr_v), 1), torch.cat((u_hat, uv), 1)

    def decode(self, L, genie_msg=None):
        return self._dec(L, self.root, genie_msg)[0]

    def channel(self, x, snr_db):
        sigma = snr_db2sigma(snr_db)
        if torch.is_tensor(sigma):
            sigma = sigma.view(-1, 1)
        y = x + sigma * torch.randn_like(x)
        return 2 * y / sigma ** 2

    def forward(self, msg, snr_db, genie=False):
        x = self.encode(msg)
        L = self.channel(x, snr_db)
        return self.decode(L, msg if genie else None)


@torch.no_grad()
def evaluate(model, snr_db, n_words=100000, batch=20000, min_blk_err=None, device='cuda', seed=None):
    model.eval()
    g = None
    if seed is not None:
        torch.manual_seed(seed)
    be = bke = tot = 0
    while tot < n_words:
        b = min(batch, n_words - tot)
        msg = torch.randint(0, 2, (b, model.k), device=device).float() * 2 - 1
        llr = model(msg, snr_db)
        err = (torch.sign(llr) != msg)
        be += err.sum().item()
        bke += err.any(1).sum().item()
        tot += b
        if min_blk_err is not None and bke >= min_blk_err and tot >= 20000:
            break
    model.train()
    return be / (tot * model.k), bke / tot, tot


def leaf_codeword_ce(collect, msg, offsets, kappa=20.0, first_only=False):
    """Scale-invariant codeword-level CE for each leaf (genie-aided inputs).
    metrics = cosine(L_leaf, codeword); CE over all leaf codewords."""
    losses = []
    for i, (nd, L) in enumerate(collect):
        if first_only and i > 0:
            break
        a0 = offsets[nd.path]
        mb = msg[:, a0:a0 + nd.k]
        Ln = L / (L.norm(dim=1, keepdim=True) + 1e-9) / math.sqrt(nd.n) * nd.n  # unit-norm scale
        if nd.r == 0:
            met = torch.stack((Ln.sum(1), -Ln.sum(1)), 1) / nd.n
            tgt = (mb[:, 0] < 0).long()
        else:
            T = fht(Ln) / nd.n                      # cosine with +H rows
            met = torch.cat((T, -T), 1)
            bits = (mb[:, 1:] < 0).long()
            idx = (bits * (2 ** torch.arange(nd.m, device=msg.device))).sum(1)
            tgt = idx + (mb[:, 0] < 0).long() * (2 ** nd.m)
        losses.append(F.cross_entropy(kappa * met, tgt, reduction='none'))
    return torch.stack(losses, 1)   # B, n_leaves


def leaf_offsets(root):
    off = {}
    def walk(nd, o):
        if nd.leaf:
            off[nd.path] = o; return
        walk(nd.u, o); walk(nd.v, o + nd.u.k)
    walk(root, 0)
    return off

In [ ]:
# ========================================
# MAIN COMPUTATION: checks + plain-Dumer waterfall
# ========================================
wf = {'snr': [], 'ber': [], 'bler': [], 'words': [], 'blk_err': []}
check_rows = []

if MODE == "compute_and_save":
    print("\n" + "=" * 70 + "\nMODE: compute_and_save\n" + "=" * 70 + "\n")
    torch.set_grad_enabled(False)
    D = NeuralRM(R, M).to(device).eval()          # defaults: enc_mode='rm', dec_mode='dumer', exact LSE, hard feedback
    print(f"RM({M},{R}): k={D.k}, n={D.n}, rate={D.k / D.n:.3f}, min distance 2^(m-r) = {2 ** (M - R)}")

    # ---- self-checks (the code must be a valid RM code and decode noiseless words exactly)
    msg = torch.randint(0, 2, (2000, D.k), device=device).float() * 2 - 1
    x = D.encode(msg)
    assert (torch.sign(D.decode(20 * x)) == msg).all(), "noiseless decoding failed"
    if D.k <= 20:                                   # exhaustive minimum-distance check for tiny codes only
        import itertools
        allm = torch.tensor(list(itertools.product([1., -1.], repeat=D.k)), device=device)
        w = (D.encode(allm) < 0).sum(1); print("minimum distance (exhaustive):", int(w[w > 0].min()))
    print("✓ noiseless decoding is exact")

    def run_point(snr, max_words, min_err, batch, seed):
        gen = torch.Generator(device=device).manual_seed(seed); sig = 10 ** (-snr / 20) / math.sqrt(2)
        be = bk = words = 0
        while bk < min_err and words < max_words:
            m_ = torch.randint(0, 2, (batch, D.k), device=device, generator=gen).float() * 2 - 1
            x_ = D.encode(m_); L = 2 * (x_ + sig * torch.randn(x_.shape, device=device, generator=gen)) / sig ** 2
            e = torch.sign(D.decode(L)) != m_
            be += e.sum().item(); bk += e.any(1).sum().item(); words += batch
        return be / (words * D.k), bk / words, words, bk

    # ---- optional comparison with the RM BERs printed in the paper
    for (pm, pr, snr), ref in PAPER_REF_BER.items():
        if (pm, pr) != (M, R): continue
        ber, bler, words, bk = run_point(snr, CHECK_WORDS if ref < 1e-2 else CHECK_WORDS // 10, 10 ** 9, BATCH, 11)
        check_rows.append((snr, ber, ref)); print(f"check RM({M},{R}) {snr:+.1f} dB: BER {ber:.3e} vs paper {ref:.2e} (ratio {ber / ref:.2f})")

    # ---- waterfall (adaptive Monte-Carlo)
    t0 = time.time()
    for si, snr in enumerate(EVAL_SNRS):
        ber, bler, words, bk = run_point(snr, MAX_WORDS, MIN_BLOCK_ERRORS, BATCH, 1000 + si)
        for k_, v_ in zip(['snr', 'ber', 'bler', 'words', 'blk_err'], [snr, ber, bler, words, bk]): wf[k_].append(v_)
        print(f"SNR {snr:+.2f} dB  BER {ber:.3e}  BLER {bler:.3e}  ({bk} block errors, {words} words)  [{time.time() - t0:.0f}s]")
    print("✓ Computation completed")

elif MODE == "analyze_only":
    print("\n" + "=" * 70 + "\nMODE: analyze_only\n" + "=" * 70 + "\n")
    d = sio.loadmat(RESULTS_DIR / "results_metrics.mat")
    wf = {k_: d[k_].ravel().tolist() for k_ in ['snr', 'ber', 'bler', 'words', 'blk_err']}
    R, M = int(d['R'].item()), int(d['M'].item()); print("Loaded previous results from", RESULTS_DIR)

## Visualization

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a_, key, name in zip(ax, ['ber', 'bler'], ['BER', 'BLER']):
    y = np.array(wf[key]); ok = y > 0
    a_.semilogy(np.array(wf['snr'])[ok], y[ok], 'k-^', linewidth=1.5, label=f'RM({M},{R}) + Dumer')
    a_.set_xlabel('SNR $E_s/N_0$ (dB)'); a_.set_ylabel(name); a_.set_title(f'RM({M},{R}): {name}', fontweight='bold')
    a_.grid(True, which='both', alpha=0.3); a_.legend()
plt.tight_layout(); plt.show()
fig.savefig(RESULTS_DIR / "results_visualization.png", dpi=PLOT_DPI, bbox_inches='tight')
print("✓ Figure saved to disk")

## Save results (.mat, .txt, config.json)

In [ ]:
if MODE == "compute_and_save":
    metrics_dict = {'R': R, 'M': M, 'seed': SEED}
    metrics_dict.update({k_: np.array(v_) for k_, v_ in wf.items()})
    lines = [f"{'=' * 70}\nPLAIN RM + DUMER BASELINE\n{'=' * 70}\n", f"Code: RM({M},{R})  k={D.k} n={D.n} rate={D.k / D.n:.3f}",
             f"Evaluation: >= {MIN_BLOCK_ERRORS} block errors per point (cap {MAX_WORDS} words)\n", f"{'SNR':>6} | {'BER':>10} | {'BLER':>10} | block errors"]
    lines += [f"{s:6.2f} | {b:10.3e} | {l:10.3e} | {int(e)}" for s, b, l, e in zip(wf['snr'], wf['ber'], wf['bler'], wf['blk_err'])]
    if check_rows:
        lines += ["", "Comparison with the RM BERs printed in the paper:"] + [f"  {s:+.1f} dB: this notebook {b:.3e}, paper {r_:.2e}, ratio {b / r_:.2f}" for s, b, r_ in check_rows]
    save_results(metrics_dict, "\n".join(lines), RESULTS_DIR); log_all_config(RESULTS_BASE_DIR)
    print("\n" + "=" * 70 + "\n✓ ALL RESULTS SAVED SUCCESSFULLY!\n" + "=" * 70)
else:
    print("analyze_only: nothing to save")